## Final Project: Analysis Notebook

This is the starting point for the *analysis notebook*. It installs and
imports every tool the course notebooks for Chapters 13 through 16 used, then
loads the corpus that `project_collect.ipynb` saved to Drive. Nothing here
calls an API; if a table is missing, the fix is in the collection notebook,
not here. Run the setup cells, check that every table loaded, and write your
findings below the last cell.

### Setup

Run all of the following before starting. These are the same setup lines
the course notebooks used, collected in one place: the `funs` module, the
UMAP and igraph packages that Chapters 15 and 16 depend on, and the spaCy
model in case you need to annotate any new text.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
! pip install umap-learn igraph --quiet

In [ ]:
! python -m spacy download en_core_web_sm

In [ ]:
from pathlib import Path

import numpy as np
import polars as pl
from plotnine import ggplot, aes, element_text, arrow
from polars import col as c

import funs
from funs import DSText, DSNetwork, umap_text

import spacy

### Loading the corpus

Mount Drive and point `CORPUS_DIR` at the directory the collection notebook
wrote. The name below matches the default `CORPUS_NAME` in that notebook;
change it if you changed that. If you did not use Drive, upload the zip with
`files.upload()`, unzip it with `! unzip -q corpus.zip -d corpus`, and set
`CORPUS_DIR` to `Path("corpus")` instead.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CORPUS_DIR = Path("/content/drive/MyDrive/corpus")

In [ ]:
pages = pl.read_parquet(CORPUS_DIR / "pages.parquet")
texts = pl.read_parquet(CORPUS_DIR / "texts.parquet")
links = pl.read_parquet(CORPUS_DIR / "links.parquet")
revisions = pl.read_parquet(CORPUS_DIR / "revisions.parquet")
pageviews = pl.read_parquet(CORPUS_DIR / "pageviews.parquet")
langlinks = pl.read_parquet(CORPUS_DIR / "langlinks.parquet")
anno = pl.read_parquet(CORPUS_DIR / "anno.parquet")
meta = pl.read_csv(CORPUS_DIR / "metadata.csv")

for name, df in [("pages", pages), ("texts", texts), ("links", links),
                 ("revisions", revisions), ("pageviews", pageviews),
                 ("langlinks", langlinks), ("anno", anno), ("meta", meta)]:
    print(f"{name:10s} {df.shape[0]:>9,} rows  {df.shape[1]:>3} columns")

In [ ]:
print((CORPUS_DIR / "MANIFEST.md").read_text())

The `links` table holds every outgoing link. The network chapter works with
the edges that stay inside the corpus, which is a join against the spine, so
that view is built here once and named `links_internal`.

In [ ]:
links_internal = (
    links
    .join(pages.select(link_title = c.doc_id, to_doc_id = c.doc_id), on=c.link_title)
    .select(c.doc_id, c.to_doc_id)
)
links_internal

### A look at each table

One line of output per table, so that a key that did not resolve or a column
that came through as the wrong type shows up now rather than in the middle
of a finding. Every table should join to `pages` on `doc_id` with no rows
lost.

In [ ]:
pages

In [ ]:
for name, df in [("texts", texts), ("links", links), ("revisions", revisions),
                 ("pageviews", pageviews), ("langlinks", langlinks), ("anno", anno)]:
    n_orphan = df.join(pages.select(c.doc_id), on=c.doc_id, how="anti").height
    print(f"{name:10s} {df['doc_id'].n_unique():>4} pages, {n_orphan} rows with a doc_id not in pages")

In [ ]:
revisions.head()

In [ ]:
pageviews.head()

In [ ]:
anno.head()

### Findings

Each finding below needs a claim, the evidence, and one sentence saying what
it is evidence of. See the project description for the full list of what the
analysis must include.

#### Finding 1